# 04 — GARCH Family Model Estimation

This notebook fits GARCH(1,1), EGARCH(1,1), and GJR-GARCH(1,1) models
to BTC log returns scaled by 100. Models are compared by AIC and BIC.

**Pipeline steps 4–5 of 7.**

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from src.data_loader import load_processed_data
from src.preprocessing import prepare_returns_for_garch
from src.model_builder import GARCHModelFactory, ModelSpec, model_comparison_table, grid_search_garch
from src.diagnostics import arch_lm_test

%matplotlib inline

## 4.1 Load and Scale Returns

In [ ]:
df = load_processed_data()
raw_returns = df["log_return"].dropna()
returns_pct = prepare_returns_for_garch(raw_returns)  # ×100
print(f"Returns range: [{returns_pct.min():.2f}, {returns_pct.max():.2f}]")

## 4.2 ARCH-LM Test (Pre-GARCH)

Confirm ARCH effects are present before fitting GARCH.

In [ ]:
lm_stat, lm_pval, arch_present = arch_lm_test(returns_pct, lags=10, verbose=True)
print(f"
ARCH effects present: {not arch_present}")

## 4.3 Baseline: GARCH(1,1) with Normal Distribution

In [ ]:
factory = GARCHModelFactory(returns_pct)
spec_base = ModelSpec("GARCH", p=1, q=1, distribution="normal")
result_base = factory.fit(spec_base)
print(result_base.summary())
print(result_base.result.summary())

## 4.4 Grid Search Over GARCH Variants

In [ ]:
all_results, comparison_table = grid_search_garch(
    returns_pct,
    model_types=["GARCH", "EGARCH", "GJR-GARCH"],
    p_range=(1, 1),
    q_range=(1, 1),
    distributions=["normal", "t", "skewt"],
)
print(comparison_table)
comparison_table.to_csv("../results/tables/garch_comparison.csv", index=False)

## 4.5 Conditional Volatility Plot

In [ ]:
cond_vol = result_base.result.conditional_volatility

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
axes[0].plot(returns_pct.index, returns_pct.values, linewidth=0.6, color="steelblue", label="Returns (×100)")
axes[0].set_title("BTC Log Returns and Conditional Volatility")
axes[0].legend()

axes[1].plot(cond_vol.index, cond_vol.values, linewidth=0.8, color="coral", label="Conditional Volatility (GARCH(1,1))")
axes[1].set_xlabel("Date")
axes[1].legend()

plt.tight_layout()
plt.savefig("../results/figures/garch_conditional_vol.png", dpi=150)
plt.show()